# Dataset Statistics/exploration (Safe to run, better run one time)

Try to understand the quality of the data better. 
In total five types of data:

    1. DMD patterns 
    2. Chromox real beam
    3. YAG real beam
    4. Chromox laser scan
    5. Yag laser scan

Beam data started from Wednesday (Chromox) (2025-11-19), then fiber state changed, then Friday and Saturday (Chromox, 2025-11-21 + 2025-11-22), Sunday (YAG, 2025-11-23), with laser scan also in Saturday and Sunday, plus DMD in the middle of sections (Many collected in 2025-11-20)


In [ ]:

from dirs import *

from xflow import SqlProvider, flow, consume, TransformRegistry as T, as_hook, compose_hooks
from xflow.utils import plot_image
import xflow.extensions.physics
from xflow.utils.io import scan_files, create_directory
from xflow.utils.sql import union_sqlite_db_tables, merge_sqlite_dbs
from xflow.extensions.physics.beam import extract_beam_parameters

from config_utils import load_config, detect_machine
from utils import *

from functools import partial
import pandas as pd
import numpy as np
import sqlite3
import json
from PIL import Image
from tqdm import tqdm

experiment_name = "CLEAR25_data_prep"  
dataset = "YAG_Laser_only" # "DMD_only" or "Chromox_only" or "Yag_only" or "Chromox_Laser_only" or "YAG_Laser_only" 

config = load_config(
    f"{experiment_name}.yaml",
    machine=machine
)

global_params = {"original_image_crop_size": None, "original_image_crop_coords": None}

# Scope 1 - DMD synthetic data and its corresponding Real data

Create such ready to use dataset for training and evaluation (could reverse the training testing logic)

In [ ]:
# ============================
# Merge entire CLEAR 2025 dataset in to a single database
# ============================
db_paths = scan_files(dirs["raw_db_dir"], extensions=[".db"], return_type="str")
merge_sqlite_dbs(db_paths, output_path=dirs["merged_db_path"], source_column="db_path")

# ============================
# Left join metadata into the big merged database to form a single complete table
# ============================
sql = """
SELECT
    d.*,
    c.experiment_description,
    c.image_source,
    c.image_device,
    c.fiber_config,
    c.camera_config,
    c.other_config
FROM mmf_dataset_metadata AS d
LEFT JOIN mmf_experiment_config AS c
  ON c.id = d.config_id
 AND c.db_path = d.db_path;
"""

with sqlite3.connect(str(dirs["merged_db_path"])) as con:
    tables_df = pd.read_sql_query(sql, con)

# optional: drop duplicate column names (e.g. both tables have "id", "db_path")
tables_df = tables_df.loc[:, ~tables_df.columns.duplicated()]
print(tables_df.shape)

In [ ]:
def hourly_counts(df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate rows by hour and return counts sorted chronologically."""
    return (
        pd.to_datetime(df["create_time"], errors="coerce")
          .dt.strftime("%Y-%m-%d %H")
          .dropna()
          .value_counts()
          .rename_axis("hour")
          .reset_index(name="count")
          .sort_values("hour")
          .reset_index(drop=True)
    )

# init once
tables_df["data_class"] = "UNCLASSIFIED"

# ============================
# DMD hourly data collection statistics
# ============================
tables_df["other_config"] = tables_df["other_config"].map(
    lambda x: x if isinstance(x, dict)
    else json.loads(x) if isinstance(x, str) and x.strip().startswith("{")
    else None
)
dmd_mask = (
    tables_df["other_config"].map(
        lambda d: isinstance(d, dict)
        and d.get("dmd_config", {}).get("type") != "DummyDMD"
    )
    & tables_df["beam_settings"].isna()
)
tables_df.loc[dmd_mask, "data_class"] = "DMD"

# ============================
# Chromox hourly data collection statistics
# ============================
tables_df.loc[
    tables_df["beam_settings"].notna() &
    tables_df["image_device"].astype(str).str.contains("Chromox", na=False),
    "data_class"
] = "CHROMOX_eBeam"

# ============================
# YAG hourly data collection statistics
# ============================
tables_df.loc[
    tables_df["beam_settings"].notna() &
    tables_df["image_device"].astype(str).str.contains("YAG", na=False),
    "data_class"
] = "YAG_eBeam"

# ============================
# Laser scans data collection statistics
# ============================
db = tables_df["db_path"].astype(str)
batch_num = pd.to_numeric(tables_df["batch"], errors="coerce")

chromox_laser_mask = db.str.contains("2025-11-22/", na=False) & batch_num.isin([1, 2, 12])  # Chromox laser
yag_laser_mask = db.str.contains("2025-11-23-morning", na=False)  # Yag 532nm laser

tables_df.loc[chromox_laser_mask, "data_class"] = "CHROMOX_LASER_SCAN"
tables_df.loc[yag_laser_mask, "data_class"] = "YAG_LASER_SCAN"

# ============================
# Assign df based on data class
# ============================
dmd_df     = tables_df[tables_df["data_class"] == "DMD"].copy()
chromox_df = tables_df[tables_df["data_class"] == "CHROMOX_eBeam"].copy()
yag_df     = tables_df[tables_df["data_class"] == "YAG_eBeam"].copy()
chrmox_laser_df = tables_df[tables_df["data_class"] == "CHROMOX_LASER_SCAN"].copy()
yag_laser_df = tables_df[tables_df["data_class"] == "YAG_LASER_SCAN"].copy()

out_1 = hourly_counts(dmd_df)
out_2 = hourly_counts(chromox_df)
out_3 = hourly_counts(yag_df)
out_4 = hourly_counts(chrmox_laser_df)
out_5 = hourly_counts(yag_laser_df)

print("Total rows kept:", 'DMD:', len(dmd_df), 'CHROMOX_eBeam:', len(chromox_df), 'YAG_eBeam:', len(yag_df), 'CHROMOX_LASER_SCAN:', len(chrmox_laser_df), 'YAG_LASER_SCAN:', len(yag_laser_df))

In [ ]:
import matplotlib.pyplot as plt

# include out_4 too
for df in (out_1, out_2, out_3, out_4, out_5):
    df["hour"] = pd.to_datetime(df["hour"])
    df.sort_values("hour", inplace=True)

# union of all active hours (including laser)
hours = sorted(set(out_1["hour"]) | set(out_2["hour"]) | set(out_3["hour"]) | set(out_4["hour"]) | set(out_5["hour"]))

# align each dataframe to same hourly axis
a = out_1.set_index("hour").reindex(hours, fill_value=0)["count"]  # DMD
b = out_2.set_index("hour").reindex(hours, fill_value=0)["count"]  # Chromox
c = out_3.set_index("hour").reindex(hours, fill_value=0)["count"]  # YAG
d = out_4.set_index("hour").reindex(hours, fill_value=0)["count"]  # Chromox Laser
e = out_5.set_index("hour").reindex(hours, fill_value=0)["count"]  # YAG Laser


x = np.arange(len(hours))
w = 0.75

fig, ax = plt.subplots(figsize=(14, 5))

# one stacked bar per hour
ax.bar(x, a, width=w, label="DMD", color="tab:blue", alpha=0.9, zorder=2)
ax.bar(x, b, width=w, bottom=a, label="Chromox", color="tab:orange", alpha=0.9, zorder=2)
ax.bar(x, c, width=w, bottom=a+b, label="YAG", color="tab:green", alpha=0.9, zorder=2)
ax.bar(x, d, width=w, bottom=a+b+c, label="Chromox Laser", color="tab:red", alpha=0.9, zorder=2)
ax.bar(x, e, width=w, bottom=a+b+c+d, label="YAG Laser", color="tab:purple", alpha=0.9, zorder=2)

# alternating day shading + day labels
days = pd.Series(hours).dt.date
day_change_idx = np.where(days != days.shift(1))[0]
total = a + b + c + d + e
ymax = total.max()

for i, start in enumerate(day_change_idx):
    end = day_change_idx[i + 1] if i + 1 < len(day_change_idx) else len(hours)
    if i % 2 == 0:
        ax.axvspan(start - 0.5, end - 0.5, color="grey", alpha=0.18, zorder=0)

    x_mid = (start + end - 1) / 2
    day_text = pd.to_datetime(hours[start]).strftime("%Y-%m-%d")
    ax.text(x_mid, ymax * 1.02, day_text, ha="center", va="bottom", fontsize=9, color="grey")

ax.set_xticks(x)
ax.set_xticklabels([h.strftime("%m-%d %H") for h in hours], rotation=45, ha="right")
ax.set_xlabel("Hour")
ax.set_ylabel("Count")
ax.set_title("Hourly count distribution (stacked)")
ax.set_ylim(0, ymax * 1.12)
ax.legend()

plt.tight_layout()
plt.show()

# Merge/Compile target .db then based on the .db create the final dataset
First select the data we want, compile to a single .db, create a column with abs path to samples, build datapipeline and go though this .db, keep the structure and save as a new ready to train folder
1. DMD only validation
2. If 1 works, DMD training + Chromox validation
3. There's two setup of MMF, one before 2025-11-20, one after. We do one after first
4. Figure out crop position

In general the valid DMD + Chromox as one 

In [ ]:
core_col_mmf_dataset_metadata = ["image_id", "is_calibration", "batch", "purpose", "db_path", "image_path", "comments", "beam_settings",
                                 "create_time", "is_saturated_ground_truth", "is_saturated_fiber_output", "config_id"]
core_col_mmf_experiment_config = ["experiment_description", "image_source", "image_device", "camera_config", "other_config"]

"""
 '2025-11-19',                          # dataset 1
 '2025-11-20',
 '2025-11-20_DMD_for_Wednesday_1',      # dataset 1
 '2025-11-20_DMD_for_Wednesday_2',      # dataset 1
 '2025-11-21',
 '2025-11-21-morning',
 '2025-11-22',
 '2025-11-22-afternoon',
 '2025-11-22-morning'
"""

if dataset == "DMD_only":
    df_to_merge = [dmd_df] 
elif dataset == "Chromox_only":
    df_to_merge = [chromox_df]
elif dataset == "Yag_only":
    df_to_merge = [yag_df]
elif dataset == "Chromox_Laser_only":
    df_to_merge = [chrmox_laser_df]
elif dataset == "YAG_Laser_only":
    df_to_merge = [yag_laser_df]

final_df = pd.concat(df_to_merge, ignore_index=True, copy=False)
final_df = final_df.loc[:, [c for c in core_col_mmf_dataset_metadata + core_col_mmf_experiment_config if c in final_df.columns]]
base = final_df["db_path"].fillna("").str.rsplit("/", n=2).str[0] + "/"
final_df["image_path"] = base + final_df["image_path"].fillna("").str.lstrip("/")
final_df["dataset"] = final_df["db_path"].astype(str).str.rsplit("/", n=3).str[-3]
final_df = final_df.drop(columns=["db_path"])
# these set are before fiber change. might be corrupted. so we just exlude them. in additoin the saved image dim is not the same.
remove_datasets = ['2025-11-19','2025-11-20_DMD_for_Wednesday_1','2025-11-20_DMD_for_Wednesday_2']  
final_df = final_df[~final_df["dataset"].isin(remove_datasets)].copy()
final_df

In [ ]:
# ============================
# Extract data sample beam parameters, just to find the cropping area
# ============================

def add_image_features(df, path_col, extractors, in_place=True):
    """
    extractors: {"new_col_1": fn1, "new_col_2": fn2, ...}
      each fn takes left-half image as ndarray and returns a value (scalar / tuple / list)
    """
    if not in_place:
        df = df.copy()

    results = {new_col: [] for new_col in extractors}

    for p in tqdm(df[path_col].tolist(), desc="images"):
        if p is None or (isinstance(p, float) and np.isnan(p)) or str(p).strip() == "":
            for new_col in results:
                results[new_col].append(np.nan)
            continue

        try:
            with Image.open(p) as im:
                w, h = im.size
                left_np = np.asarray(im.crop((0, 0, w // 2, h)))
            for new_col, fn in extractors.items():
                results[new_col].append(fn(left_np))
        except Exception:
            for new_col in results:
                results[new_col].append(np.nan)

    for new_col, vals in results.items():
        df[new_col] = vals

    return df

extract_gaussian = partial(extract_beam_parameters, method="gaussian")
extract_moments = partial(extract_beam_parameters, method="moments")

final_df = add_image_features(
    final_df,
    path_col="image_path",
    extractors={"gaussian_beam_params": extract_gaussian, "moments_beam_params": extract_moments},
    in_place=True,
)

final_df = final_df.applymap(lambda x: str(x) if isinstance(x, (dict, list, tuple)) else x)
with sqlite3.connect(dirs["final_merged_db_path"]) as conn:
    final_df.to_sql("mmf_dataset_metadata", conn, if_exists="replace", index=False)

In [ ]:
# ============================
# No need to rerun, just read the final merged database and continue analysis, core dataframe to load for downstream
# ===========================

table = "mmf_dataset_metadata"

with sqlite3.connect(dirs["final_merged_db_path"]) as conn:
    final_df = pd.read_sql_query(f'SELECT * FROM "{table}"', conn)

print(final_df.shape)

# Try to define the crop area

In [ ]:
# ============================
# Plot beam centroids and crop square, Define the cropping area, deprecated
# ============================

import ast
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

vec_col = "moments_beam_params"
ds_col  = "dataset"

W, H = 1920, 1200          # image size
R = 200                    # half-side length in pixels (distance from centroid to each edge)     230

def parse_vec4(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    if isinstance(v, (list, tuple, np.ndarray)):
        a = list(v)
    else:
        s = str(v).strip()
        try:
            a = ast.literal_eval(s)
        except Exception:
            s = s.strip("[]()")
            a = [t for t in s.replace(",", " ").split() if t]
    a = [float(x) for x in a]
    return a if len(a) >= 2 else None

def centroid_xy_norm(xy_series, W=None, H=None, R=None, verbose=True):
    """
    xy_series: Series of list-like [x, y, ...] in normalized coords.
    R: half-side length in px (crop size is 2R x 2R).
    Returns:
      (cx_norm, cy_norm), (cx_px, cy_px), ((tl_px), (br_px)), ((tl_int), (br_int_excl))
    """
    arr = np.array([v[:2] for v in xy_series.dropna().to_list()], dtype=float)
    if arr.size == 0:
        return None

    cx_norm, cy_norm = arr[:, 0].mean(), arr[:, 1].mean()
    cx_px,  cy_px  = cx_norm * W, cy_norm * H

    tl = (cx_px - R, cy_px - R)
    br = (cx_px + R, cy_px + R)

    # integer crop box with exact size (2R x 2R); br_int is EXCLUSIVE
    R_int = int(R)
    tl_int = (int(round(tl[0])), int(round(tl[1])))
    br_int = (tl_int[0] + 2*R_int, tl_int[1] + 2*R_int)

    if verbose:
        print(f"centroid (px): ({cx_px:.2f}, {cy_px:.2f})")
        print(f"square TL, BR (px): ({tl[0]:.2f}, {tl[1]:.2f}), ({br[0]:.2f}, {br[1]:.2f})")
        print(f"square TL, BR (int, size={2*R_int}x{2*R_int} px): {tl_int}, {br_int}")
        global_params["original_image_crop_size"] = (2*R_int, 2*R_int)
        global_params["original_image_crop_coords"] = (tl_int, br_int)

    return (cx_norm, cy_norm), (cx_px, cy_px), (tl, br)


xy = final_df[vec_col].apply(parse_vec4)
mask = xy.notna() & final_df[ds_col].notna()

# normalized points
x_norm = xy[mask].apply(lambda a: a[0]).to_numpy()
y_norm = xy[mask].apply(lambda a: a[1]).to_numpy()
ds = pd.Categorical(final_df.loc[mask, ds_col].astype(str))

# centroid in normalized coords
c = centroid_xy_norm(xy[mask], W=W, H=H, R=R, verbose=True)
(cx_norm, cy_norm), (cx, cy), (tl, br) = c

# convert to pixel coords only for plotting
x = x_norm * W
y = y_norm * H
cx = cx_norm * W
cy = cy_norm * H

cmap = plt.get_cmap("tab10")
colors = [cmap(i % cmap.N) for i in ds.codes]

# Match Section 2 visual proportions
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(x, y, s=0.5, c=colors, alpha=0.2, edgecolors="none")

# Smaller legend handles
for i, name in enumerate(ds.categories):
    ax.scatter([], [], s=12, color=cmap(i % cmap.N), label=name)

# Put legend inside, compact
ax.legend(
    title=ds_col,
    loc="upper right",
    bbox_to_anchor=(0.98, 0.98),
    fontsize=7,
    title_fontsize=8,
    frameon=True,
    framealpha=0.85,
    borderpad=0.25,
    labelspacing=0.25,
    handlelength=1.0,
    handletextpad=0.4,
    markerscale=0.9,
)


# Match square + center styling from Section 2
ax.add_patch(Rectangle((cx - R, cy - R), 2*R, 2*R, fill=False, linewidth=1, edgecolor="red"))
ax.scatter([cx], [cy], marker=".", s=10, c="red")

# Match ticks/labels style from Section 2
xt = np.linspace(0, W - 1, 6)
yt = np.linspace(0, H - 1, 6)
ax.set_xticks(xt)
ax.set_yticks(yt)
ax.set_xticklabels(np.round(np.linspace(0, W, 6)).astype(int))
ax.set_yticklabels(np.round(np.linspace(0, H, 6)).astype(int))

ax.set_xlim(0, W - 1)
ax.set_ylim(H - 1, 0)  # make y-axis direction visually match imshow(origin="upper")
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x (px)")
ax.set_ylabel("y (px)")
ax.set_title(f"RMS beam centroids | crop {2*R}x{2*R}")

fig.tight_layout()
plt.show()


In [ ]:
# ============================
# Alternative Crop logic 2: Maximum Intensity Projection (MIP) of all images
# ============================

from PIL import Image
import numpy as np
from tqdm.auto import tqdm

# Inputs
image_to_stack = final_df["image_path"].tolist()
raw_npz_path = dirs["globel_save_to"] + "mip_raw.npz"  # or use "stacked_raw.npz"

# Build Maximum Intensity Projection (raw linear values)
mip = None
used = 0

for p in tqdm(image_to_stack, desc="images"):
    if p is None or (isinstance(p, float) and np.isnan(p)) or str(p).strip() == "":
        continue
    try:
        with Image.open(p) as im:
            w, h = im.size
            left_np = np.asarray(im.crop((0, 0, w // 2, h)).convert("L"), dtype=np.float64)

        if mip is None:
            mip = left_np
        elif left_np.shape == mip.shape:
            mip = np.maximum(mip, left_np)
        else:
            continue

        used += 1
    except Exception:
        continue

if mip is None:
    raise ValueError("No valid images found.")

# Save raw only (unchanged linear values)
np.savez_compressed(
    raw_npz_path,
    mip=mip,
    used=np.array([used], dtype=np.int64),
)

print("Saved:", raw_npz_path, "| mip shape:", mip.shape, "| used:", used)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from mpl_toolkits.axes_grid1 import make_axes_locatable

# Inputs
raw_npz_path = dirs["globel_save_to"] + "mip_raw.npz"  # same as block 1
R = 200
S = 2 * R
display_mode = "linear"   # "linear" or "log" (for plotting only)

# Load raw unchanged MIP
data = np.load(raw_npz_path, allow_pickle=True)
mip = data["mip"]
used = int(data["used"][0])

# Fit on linear raw data
H, W = mip.shape
if S > H or S > W:
    raise ValueError(f"Square size {S}x{S} is larger than image {W}x{H}.")

I = np.pad(mip, ((1, 0), (1, 0)), mode="constant").cumsum(0).cumsum(1)
win_sum = I[S:, S:] - I[:-S, S:] - I[S:, :-S] + I[:-S, :-S]

top, left = np.unravel_index(np.argmax(win_sum), win_sum.shape)
cx, cy = left + R, top + R

print(f"Used images: {used}")
print(f"Best square center (px): ({cx}, {cy})")
print(f"Best square TL, BR (px): ({left}, {top}), ({left + S}, {top + S})")
print(f"Best square size: {S}x{S}, score={float(win_sum[top, left]):.6f}")

# Plot only
vis = mip if display_mode == "linear" else np.log1p(mip)

fig, ax = plt.subplots(figsize=(6, 6))
im_plot = ax.imshow(vis, cmap="viridis", origin="upper", aspect="equal")

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="4%", pad=0.05)
fig.colorbar(im_plot, cax=cax, label="intensity" if display_mode == "linear" else "log(intensity)")

ax.add_patch(Rectangle((left, top), S, S, fill=False, linewidth=2, edgecolor="red"))
ax.scatter([cx], [cy], marker=".", s=10, c="red")

xt = np.linspace(0, W - 1, 6)
yt = np.linspace(0, H - 1, 6)
ax.set_xticks(xt)
ax.set_yticks(yt)
ax.set_xticklabels(np.round(np.linspace(0, W, 6)).astype(int))
ax.set_yticklabels(np.round(np.linspace(0, H, 6)).astype(int))

ax.set_xlabel("x (px)")
ax.set_ylabel("y (px)")
ax.set_title(f"MIP image | best {S}x{S} window")

fig.tight_layout()
plt.show()

global_params["original_image_crop_size"] = (2 * int(R), 2 * int(R))
global_params["original_image_crop_coords"] = ((int(left), int(top)), (int(left + S), int(top + S)))
print("global_params:", global_params)

In [ ]:
# ============================
# Alternative Crop logic 3: Direct stack images in log scale
# ============================

from pathlib import Path
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from mpl_toolkits.axes_grid1 import make_axes_locatable

# Settings
display_mode = "log"   # "linear" or "log"
image_to_stack = final_df["image_path"].tolist()[:]   # keep your original behavior

# Output paths
out_dir = dirs["globel_save_to"]
render_png_path = out_dir + "stacked_vis.png"
raw_npz_path = out_dir + "stacked_raw.npz"

# Build stack
stack_sum = None
used = 0

for p in tqdm(image_to_stack, desc="images"):
    if p is None or (isinstance(p, float) and np.isnan(p)) or str(p).strip() == "":
        continue

    try:
        with Image.open(p) as im:
            w, h = im.size
            left_np = np.asarray(im.crop((0, 0, w // 2, h)).convert("L"), dtype=np.float64)

        if stack_sum is None:
            stack_sum = np.zeros_like(left_np, dtype=np.float64)

        stack_sum += left_np
        used += 1

    except Exception:
        continue

if stack_sum is None:
    raise ValueError("No valid images found.")

# Build visualization array
if display_mode == "linear":
    vis = stack_sum
    title = f"original image stacked sum (linear)"
elif display_mode == "log":
    vis = np.log1p(stack_sum)
    title = f"original image stacked sum (log)"
else:
    raise ValueError("display_mode must be 'linear' or 'log'.")

# Save raw data unchanged
np.savez_compressed(
    raw_npz_path,
    stack_sum=stack_sum,
    vis=vis,
    used=np.array([used], dtype=np.int64),
    display_mode=np.array([display_mode]),
)

# Plot and save high-res PNG
H, W = vis.shape
fig, ax = plt.subplots(figsize=(6, 6))
im_plot = ax.imshow(vis, cmap="viridis", origin="upper", aspect="equal")

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="4%", pad=0.05)
fig.colorbar(im_plot, cax=cax, label="intensity")

xt = np.linspace(0, W - 1, 6)
yt = np.linspace(0, H - 1, 6)
ax.set_xticks(xt)
ax.set_yticks(yt)
ax.set_xticklabels(np.round(np.linspace(0, W, 6)).astype(int))
ax.set_yticklabels(np.round(np.linspace(0, H, 6)).astype(int))

ax.set_xlabel("x (px)")
ax.set_ylabel("y (px)")
ax.set_title(title)

fig.tight_layout()
fig.savefig(render_png_path, dpi=400, bbox_inches="tight")
plt.show()

print("Saved:", render_png_path)
print("Saved:", raw_npz_path)

In [ ]:
# ============================
# Section 2: Load saved data, estimate best square, plot + save overlay
# ============================

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from mpl_toolkits.axes_grid1 import make_axes_locatable

# Inputs
out_dir = dirs["globel_save_to"]
raw_npz_path = out_dir + "stacked_raw.npz"

# User-provided half-side length
R = 200
S = 2 * R

# Load
data = np.load(raw_npz_path, allow_pickle=True)
vis = data["vis"]
used = int(data["used"][0])
display_mode = str(data["display_mode"][0])

H, W = vis.shape
if S > H or S > W:
    raise ValueError(f"Square size {S}x{S} is larger than image {W}x{H}.")

# Max-sum SxS window via integral image
I = np.pad(vis, ((1, 0), (1, 0)), mode="constant").cumsum(0).cumsum(1)
win_sum = I[S:, S:] - I[:-S, S:] - I[S:, :-S] + I[:-S, :-S]

top, left = np.unravel_index(np.argmax(win_sum), win_sum.shape)
best_sum = float(win_sum[top, left])

cx = left + R
cy = top + R
tl = (left, top)
br = (left + S, top + S)  # exclusive end in array convention

print(f"Best square center (px): ({cx}, {cy})")
print(f"Best square TL, BR (px): {tl}, {br}")
print(f"Best square size: {S}x{S}, score={best_sum:.6f}")

# Plot overlay
fig, ax = plt.subplots(figsize=(6, 6))
im_plot = ax.imshow(vis, cmap="viridis", origin="upper", aspect="equal")

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="4%", pad=0.05)
fig.colorbar(im_plot, cax=cax, label="intensity")

ax.add_patch(Rectangle((left, top), S, S, fill=False, linewidth=2, edgecolor="red"))
ax.scatter([cx], [cy], marker=".", s=10, c="red")

xt = np.linspace(0, W - 1, 6)
yt = np.linspace(0, H - 1, 6)
ax.set_xticks(xt)
ax.set_yticks(yt)
ax.set_xticklabels(np.round(np.linspace(0, W, 6)).astype(int))
ax.set_yticklabels(np.round(np.linspace(0, H, 6)).astype(int))

ax.set_xlabel("x (px)")
ax.set_ylabel("y (px)")
ax.set_title(f"stacked image ({display_mode}) | best {S}x{S} window")

fig.tight_layout()
plt.show()

# save fitted square to global params (all ints)
R_int = int(R)
tl_int = (int(left), int(top))
br_int = (int(left + S), int(top + S))  # bottom-right exclusive

global_params["original_image_crop_size"] = (2 * R_int, 2 * R_int)
global_params["original_image_crop_coords"] = (tl_int, br_int)

print("global_params:", global_params)

# Produce the dataset

In [ ]:
from pathlib import Path
from functools import partial
import sqlite3
import gc

# Add one extra column to check and exclude the samples that are 1. original beam not inside the crop 2. two camera are not synched well. 
# By introducing coupling rate (total output intensity / total input intensity), so if it is too low or too high we can filter it out.

global_params["original_image_crop_coords"] = ((668, 411), (1068, 811))   # Manually overwriting if necessary.

params = {
    # DMD only: [[742, 341], [1202, 801]]
    # Chromox only: [[758, 389], [1198, 829]],
    # Yag only: [[821, 481], [1077, 737]], 
    "crop_gt":  global_params["original_image_crop_coords"],
    "crop_fo": [[360, 0], [1560, 1200]],
    "inp_size": (256, 256),
    "out_size": (256, 256),
}

work_df = final_df.iloc[:].copy().reset_index(drop=True)

image_paths = work_df["image_path"].tolist()
n = len(image_paths)

# 1. Simplified to empty lists
coupling_ratio = []
split_img1_stats = []
split_img2_stats = []
is_saturated_ground_truth = []
is_saturated_fiber_output = []

# 2. Tap function simply appends what it gets or None on exception
def stats_tap(eps=1e-12):
    def _fn(payload):
        img1, img2, out_path = payload  # img1: ground truth, img2: fiber output
        try:
            cr = T.get("torch_sum_ratio")((img2, img1), eps=eps)
            st1 = str(T.get("torch_image_stats")(img1))
            st2 = str(T.get("torch_image_stats")(img2))
        except Exception as e:
            print(f"[stats_tap] Failed for out_path={out_path}: {e}")
            cr = None
            st1 = None
            st2 = None
        coupling_ratio.append(cr)
        split_img1_stats.append(st1)
        split_img2_stats.append(st2)
        is_saturated_ground_truth.append(1 if img1 is not None and img1.max() >= 255 else 0)
        is_saturated_fiber_output.append(1 if img2 is not None and img2.max() >= 255 else 0)
        return payload
    return _fn

samples = [(p, p) for p in image_paths]
results = list(
    flow(
        samples,
        [T.get("torch_load_image"), T.get("extract_stem")],
        [None, partial(T.get("add_parent_dir"), parent_dir=dirs[dataset]["output_dataset_dir"])],
        [None, partial(T.get("add_suffix"), suffix=".png")],
        # [None, T.get("debug_print")],
        # [T.get("torch_to_grayscale"), None],
        # [T.get("torch_remap_range"), None],
        [partial(T.get("torch_split_width"), swap=True), None],
        [
            partial(T.get("torch_crop_area"), points=params["crop_fo"]),
            partial(T.get("torch_crop_area"), points=params["crop_gt"]),
            None,
        ],
        [
            partial(T.get("torch_resize"), size=params["inp_size"], interpolation="bilinear"),
            partial(T.get("torch_resize"), size=params["out_size"], interpolation="bilinear"),
            None,
        ],
        partial(T.get("reorder"), order=[1, 0, 2]),  # 3-element payload: (img1, img2, out_path)
        stats_tap(eps=1e-12),
        [consume(2, lambda imgs: T.get("join_image")(list(imgs), layout=(1, 2))), None],
        T.get("save_image"),
        
        progress=True,
        desc="Processing images",
        skip_errors=True,
    )
)

assert len(coupling_ratio) == n
assert len(split_img1_stats) == n
assert len(split_img2_stats) == n

work_df["coupling_ratio"] = coupling_ratio
work_df["ground_truth_stats"] = split_img1_stats
work_df["fiber_output_stats"] = split_img2_stats

work_df["original_image_crop_size"] = str((params["crop_gt"][1][0] - params["crop_gt"][0][0], params["crop_gt"][1][1] - params["crop_gt"][0][1]))
work_df["original_image_crop_coords"] = str(params["crop_gt"])

work_df["is_saturated_ground_truth"] = is_saturated_ground_truth
work_df["is_saturated_fiber_output"] = is_saturated_fiber_output

df_save = work_df.copy()
s = df_save["image_path"]
df_save["image_path"] = s.where(
    s.isna(),
    "dataset/" + s.str.replace(r"^.*[\\/]", "", regex=True),
)

subset_db_path = str(Path(dirs[dataset]["output_db_dir"]).with_name("dataset_meta.db"))
create_directory(str(Path(subset_db_path).parent))
conn = None
try:
    conn = sqlite3.connect(subset_db_path, timeout=30)
    df_save.to_sql("mmf_dataset_metadata", conn, if_exists="replace", index=False)
    conn.commit()
finally:
    if conn is not None:
        conn.close()
    del conn
    gc.collect()

## quick test of the created basis pipeline before HPC
Once the processed dataset is created, could directly run below (and the first config cell) cells without run anything else

In [ ]:
from xflow.extensions.physics.pipeline import CachedBasisPipeline, IndexCombinator
from xflow.data import build_transforms_from_config, PyTorchPipeline
from xflow.extensions.physics import pattern_gen

config = load_config(
    f"{experiment_name}.yaml",
    machine=machine
)

# training set (basis -> generative pipeline)
train_provider = SqlProvider(
    sources={"connection": dirs[dataset]["output_db_dir"], "sql": config["sql"]["train"]}, output_config={'list': "image_path"}
)
# evaluation set (validation + test, using real beam pattern loaded on the DMD)
eval_provider = SqlProvider(
    sources={"connection": dirs[dataset]["output_db_dir"], "sql": config["sql"]["eval"]}, output_config={'list': "image_path"}
)

background_provider = SqlProvider(
    sources={"connection": dirs[dataset]["output_db_dir"], "sql": config["sql"]["background"]}, output_config={'list': "image_path"}
)

# create data pipelines for ML training and evaluation
config["data"]["transforms"]["torch"].insert(0, {
    "name": "add_parent_dir",
    "params": {
        "parent_dir": dirs[dataset]["dataset_dir"]
    }
})
transforms = build_transforms_from_config(config["data"]["transforms"]["torch"])

canvas = pattern_gen.DynamicPatterns(256, 256)
canvas.set_postprocess_fns([T.get("remap_range"), partial(T.get("resize"), size=(32,32), interpolation="bilinear")])
canvas._distributions = [pattern_gen.StaticGaussianDistribution(canvas) for _ in range(100)]
canvas.set_threshold(10)
# (std_1=0.03, std_2=0.2, max_intensity=100, fade_rate=0.96, distribution='other'
stream = canvas.pattern_stream(std_1=0.02, std_2=0.08, max_intensity=100, fade_rate=0.98, distribution='other') 

# Option: for using runtime sample to background subtract
background_pipeline = PyTorchPipeline(
    background_provider, 
    transforms[:3],
).to_memory_dataset()
runtime_transforms = [partial(T.get("torch_subtract_tensor"), subtract_tensor=next(iter(background_pipeline))[0]), T.get("torch_clip_below_zero")]
# transforms[3:3] = runtime_transforms

# ======== random combinator using index + SGM ========
combinator = IndexCombinator(
    pattern_provider=stream,
    post_transforms= build_transforms_from_config(config["data"]["post_transforms"]["torch"]),
)

val_provider, test_provider = eval_provider.split(config["data"]["val_test_split"])
train_pipeline = CachedBasisPipeline(
    train_provider, 
    combinator=combinator, 
    transforms=transforms, 
    num_samples=1000, 
    seed=42, 
    eager=True,
)

val_pipeline = PyTorchPipeline(
    val_provider, 
    transforms[:-1],
).to_memory_dataset(config["data"]["dataset_ops"])   # testset data do not need thresholding since it is to remove stacking noise?

test_pipeline = PyTorchPipeline(
    test_provider, 
    transforms[:-1],
).to_memory_dataset(config["data"]["dataset_ops"])

In [ ]:
# Each dataset (train, val, test) takes some random samples for visualization and sanity check 
# Specifically for the training set, also print out the basis combination info
for sample in train_pipeline:
    record = combinator.last_record
    # for idx, coef in zip(record.indices, record.coefficients):
    #     basis_item = train_pipeline.get_basis(idx)
    #     print(f"  basis[{idx}] * {coef:.2f}")
    left_parts, right_parts = sample
    plot_image(left_parts[0], title="Input (Fiber Output)", figsize=(6,6))
    plot_image(right_parts[0], title="Ground Truth", figsize=(6,6))
    break

for left_parts, right_parts in val_pipeline:
    plot_image(left_parts[0], title="Input (Fiber Output)", figsize=(6,6))
    plot_image(right_parts[0], title="Ground Truth", figsize=(6,6))
    break
# for left_parts, right_parts in test_pipeline:
#     plot_image(left_parts[0], title="Input (Fiber Output)", figsize=(6,6))
#     plot_image(right_parts[0], title="Ground Truth", figsize=(6,6))
#     break

# TODO: index scale factor is not totally correct, since the peak intensity of basis is not exactly 1.0
# TODO: noise problem emerges, when stacking too many basis patterns together, need to investigate further

# Scope 2 - Real data only with data augmentation pipeline (super position)
Create such ready to use dataset for training and evaluation